In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
csv_path = Path("alpha_theta_grouped.csv")
df = pd.read_csv(csv_path)

print(df.head())
print(df["split"].unique())
print(df["group_type"].unique())

In [ ]:
# Use the combined fit across all splits
df_all = df[df["split"] == "all"].copy()

# Yearly fits
df_year = df_all[df_all["group_type"] == "year"].copy()
df_year["calendar_year"] = df_year["calendar_year"].astype(int)
df_year = df_year.sort_values("calendar_year").reset_index(drop=True)

# Season-year fits
df_sy = df_all[df_all["group_type"] == "year_season"].copy()

# Parse keys like "2015_DJF"
df_sy[["season_year", "season"]] = df_sy["group_key"].str.split("_", expand=True)
df_sy["season_year"] = df_sy["season_year"].astype(int)

season_order = {"DJF": 0, "MAM": 1, "JJA": 2, "SON": 3}
df_sy["season_order"] = df_sy["season"].map(season_order)

df_sy = df_sy.sort_values(["season_year", "season_order"]).reset_index(drop=True)

print(df_year[["calendar_year", "alpha", "theta_deg"]])
print(df_sy[["group_key", "alpha", "theta_deg"]].head(12))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_year["calendar_year"], df_year["alpha"], marker="o")
ax.set_xlabel("Year")
ax.set_ylabel("Alpha")
ax.set_title("Alpha vs year")
ax.grid(True)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_year["calendar_year"], df_year["theta_deg"], marker="o")
ax.set_xlabel("Year")
ax.set_ylabel("Theta [deg]")
ax.set_title("Theta vs year")
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
x = range(len(df_sy))
labels = df_sy["group_key"]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(x, df_sy["alpha"], marker="o")
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_xlabel("Season-year")
ax.set_ylabel("Alpha")
ax.set_title("Alpha vs season-year")
ax.grid(True)
plt.tight_layout()
plt.show()

x = range(len(df_sy))
labels = df_sy["group_key"]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(x, df_sy["theta_deg"], marker="o")
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_xlabel("Season-year")
ax.set_ylabel("Theta [deg]")
ax.set_title("Theta vs season-year")
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def plot_metric_by_year(df, metric="alpha", splits=("train", "val", "test"), ylabel=None):
    fig, ax = plt.subplots(figsize=(8, 4))

    prev_d = None

    for split in splits:
        d = df[(df["split"] == split) & (df["group_type"] == "year")].copy()
        if d.empty:
            continue

        d["calendar_year"] = d["calendar_year"].astype(int)
        d = d.sort_values("calendar_year").reset_index(drop=True)

        if prev_d is None or prev_d.empty:
            # train: line + markers on all points
            line = ax.plot(
                d["calendar_year"],
                d[metric],
                marker="o",
                label=split,
            )[0]
        else:
            # prepend last point from previous split so the line connects
            prev_point = prev_d.tail(1)
            d_plot = pd.concat([prev_point, d], ignore_index=True)

            # draw connecting line first
            line = ax.plot(
                d_plot["calendar_year"],
                d_plot[metric],
                marker=None,
                label=split,
            )[0]

            # draw markers only for the actual current split points, in same color
            ax.plot(
                d["calendar_year"],
                d[metric],
                linestyle="None",
                marker="o",
                color=line.get_color(),
            )

        prev_d = d

    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel if ylabel is not None else metric)
    ax.set_title(f"{metric} vs year")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric_by_year(df, metric="alpha", ylabel="Alpha")
plot_metric_by_year(df, metric="theta_deg", ylabel="Theta [deg]")

In [ ]:
from matplotlib.lines import Line2D
import pandas as pd
import matplotlib.pyplot as plt


def plot_alpha_theta_by_year(
    df,
    splits=("train", "val", "test"),
    alpha_color="tab:blue",
    theta_color="tab:orange",
    figsize=(9, 5),
    exclude_years=(2014,),
):
    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx()

    style_map = {
        "train": {"linestyle": "-",  "marker": "o"},
        "val":   {"linestyle": "--", "marker": "s"},
        "test":  {"linestyle": ":",  "marker": "^"},
    }

    prev_d = None

    for split in splits:
        d = df[(df["split"] == split) & (df["group_type"] == "year")].copy()
        if d.empty:
            continue

        d["calendar_year"] = d["calendar_year"].astype(int)
        d = d[~d["calendar_year"].isin(exclude_years)]
        d = d.sort_values("calendar_year").reset_index(drop=True)

        if d.empty:
            continue

        ls = style_map.get(split, {"linestyle": "-", "marker": "o"})["linestyle"]
        mk = style_map.get(split, {"linestyle": "-", "marker": "o"})["marker"]

        if prev_d is None or prev_d.empty:
            ax1.plot(
                d["calendar_year"],
                d["alpha"],
                color=alpha_color,
                linestyle=ls,
                marker=mk,
            )
            ax2.plot(
                d["calendar_year"],
                d["theta_deg"],
                color=theta_color,
                linestyle=ls,
                marker=mk,
            )
        else:
            prev_point = prev_d.tail(1)
            d_plot = pd.concat([prev_point, d], ignore_index=True)

            ax1.plot(
                d_plot["calendar_year"],
                d_plot["alpha"],
                color=alpha_color,
                linestyle=ls,
                marker=None,
            )
            ax1.plot(
                d["calendar_year"],
                d["alpha"],
                color=alpha_color,
                linestyle="None",
                marker=mk,
            )

            ax2.plot(
                d_plot["calendar_year"],
                d_plot["theta_deg"],
                color=theta_color,
                linestyle=ls,
                marker=None,
            )
            ax2.plot(
                d["calendar_year"],
                d["theta_deg"],
                color=theta_color,
                linestyle="None",
                marker=mk,
            )

        prev_d = d

    ax1.set_xlabel("Year")
    ax1.set_ylabel("Alpha", color=alpha_color)
    ax2.set_ylabel("Theta [deg]", color=theta_color)

    ax1.tick_params(axis="y", labelcolor=alpha_color)
    ax2.tick_params(axis="y", labelcolor=theta_color)

    ax1.set_title("Alpha and theta vs year")
    ax1.grid(True)

    parameter_handles = [
        Line2D([0], [0], color=alpha_color, lw=2, label="Alpha"),
        Line2D([0], [0], color=theta_color, lw=2, label="Theta"),
    ]

    split_handles = []
    for split in splits:
        if split in style_map:
            split_handles.append(
                Line2D(
                    [0], [0],
                    color="black",
                    linestyle=style_map[split]["linestyle"],
                    marker=style_map[split]["marker"],
                    lw=2,
                    label=split,
                )
            )

    legend1 = ax1.legend(handles=parameter_handles, loc="upper left", title="Parameter")
    ax1.add_artist(legend1)
    ax1.legend(handles=split_handles, loc="lower right", title="Split")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_alpha_theta_by_year(df, exclude_years=(2014,))

In [ ]:
def plot_metric_by_season(df, metric="alpha", splits=("train", "val", "test"), ylabel=None):
    fig, ax = plt.subplots(figsize=(7, 4))

    season_order = ["DJF", "MAM", "JJA", "SON"]

    for split in splits:
        d = df[(df["split"] == split) & (df["group_type"] == "season")].copy()
        if d.empty:
            continue

        d["season"] = pd.Categorical(d["season"], categories=season_order, ordered=True)
        d = d.sort_values("season").reset_index(drop=True)

        ax.plot(d["season"], d[metric], marker="o", label=split)

    ax.set_xlabel("Season")
    ax.set_ylabel(ylabel if ylabel is not None else metric)
    ax.set_title(f"{metric} vs season")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric_by_season(df, metric="alpha", ylabel="Alpha")
plot_metric_by_season(df, metric="theta_deg", ylabel="Theta [deg]")

In [ ]:
def prepare_year_season(df_in, split="all"):
    d = df_in[(df_in["split"] == split) & (df_in["group_type"] == "year_season")].copy()
    d[["season_year", "season"]] = d["group_key"].str.split("_", expand=True)
    d["season_year"] = d["season_year"].astype(int)

    season_order = {"DJF": 0, "MAM": 1, "JJA": 2, "SON": 3}
    d["season_order"] = d["season"].map(season_order)

    d = d.sort_values(["season_year", "season_order"]).reset_index(drop=True)
    return d


def plot_metric_by_year_season(df_in, metric="alpha", split="all", ylabel=None):
    d = prepare_year_season(df_in, split=split)
    x = range(len(d))

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(x, d[metric], marker="o")
    ax.set_xticks(list(x))
    ax.set_xticklabels(d["group_key"], rotation=45, ha="right")
    ax.set_xlabel("Season-year")
    ax.set_ylabel(ylabel if ylabel is not None else metric)
    ax.set_title(f"{metric} vs season-year ({split})")
    ax.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric_by_year_season(df, metric="alpha", split="all", ylabel="Alpha")
plot_metric_by_year_season(df, metric="theta_deg", split="all", ylabel="Theta [deg]")

In [ ]:
def plot_metric_by_year_season_colored(df_in, metric="alpha", split="all", ylabel=None):
    d = prepare_year_season(df_in, split=split)

    color_map = {
        "DJF": "tab:blue",
        "MAM": "tab:green",
        "JJA": "tab:orange",
        "SON": "tab:purple",
    }

    fig, ax = plt.subplots(figsize=(14, 4))

    for season in ["DJF", "MAM", "JJA", "SON"]:
        ds = d[d["season"] == season]
        ax.plot(ds.index, ds[metric], marker="o", label=season, color=color_map[season])

    ax.set_xticks(d.index)
    ax.set_xticklabels(d["group_key"], rotation=45, ha="right")
    ax.set_xlabel("Season-year")
    ax.set_ylabel(ylabel if ylabel is not None else metric)
    ax.set_title(f"{metric} vs season-year ({split})")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric_by_year_season_colored(df, metric="alpha", split="all", ylabel="Alpha")
plot_metric_by_year_season_colored(df, metric="theta_deg", split="all", ylabel="Theta [deg]")